### Credit Risk Prediction

#### 1. Problem Statement

A financial institution wants to reduce loan defaults while maximizing approvals.
The objective is to predict loan default risk using customer financial and demographic data, enabling better lending decisions and minimizing financial losse

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

#### 2. Data Understanding & Cleaning
The dataset contains customer financial, demographic, and loan-related features, with a binary target indicating default. Initial exploration focused on identifying data types, distributions, and class imbalance

Data was checked for duplicates and missing values. Categorical variables were standardized and prepared for encoding to ensure consistency

In [ ]:
df=pd.read_csv("/kaggle/input/datasets/nikhil1e9/loan-default/Loan_default.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

No significant missing values were observed, so minimal imputation was required.”

In [ ]:
df["Default"].value_counts()
df["Default"].value_counts(normalize=True)

#### 3. Data Preprocessing

In [ ]:
df.duplicated().sum()

In [ ]:
for col in df.select_dtypes(include="object").columns:
    print(col,df[col].unique())

In [ ]:
numeric_df=df.select_dtypes(include=["number"])

plt.figure(figsize=(10,8))
sns.heatmap(numeric_df.corr(),annot=True,cmap="coolwarm")
plt.show()

The dataset shows low linear correlation with the target, indicating that default risk is likely influenced by complex, non-linear relationships. Therefore, tree-based models like Random Forest or XGBoost may perform better.

#### 4. Exploratory Data Analysis (EDA)

In [ ]:
sns.countplot(data=df,x="Default")
plt.title("Default Distribution")
plt.show()

Default Distribution shows data imbalance.

In [ ]:
sns.boxplot(data=df,x="Default",y="Income")
plt.title("Income vs Default")
plt.show()

Customers with lower income show a higher likelihood of default or People with higher incomes are generally more likely to pay back their loans

In [ ]:
sns.boxplot(x='Default', y='CreditScore', data=df)
plt.title("Credit Score vs Default")
plt.show()

Customers with lower credit scores exhibit a higher probability of default.

In [ ]:
sns.boxplot(x='Default', y='LoanAmount', data=df)
plt.title("Loan Amount vs Default")
plt.show()


Bigger loans leads to higher risk + People with higher incomes are generally more likely to pay back their loans

In [ ]:
sns.countplot(x='EmploymentType', hue='Default', data=df)
plt.title("Employment Type vs Default")
plt.show()

Unemployed leads to high default

In [ ]:
sns.boxplot(x='Default', y='InterestRate', data=df)
plt.title("Interest Rate vs Default")
plt.show()

Higher interest leads to higher risk

In [ ]:
sns.boxplot(x='Education', y='InterestRate', data=df)
plt.title("Interest Rate vs Default")
plt.show()

#### 5. Feature Engineering & Encoding

In [ ]:
df=df.drop("LoanID",axis=1)

In [ ]:
binary_cols=["HasMortgage","HasDependents","HasCoSigner"]
for col in binary_cols:
    df[col]=df[col].map({"Yes":1,"No":0})

In [ ]:
df=pd.get_dummies(df,drop_first=True)

In [ ]:
X=df.drop("Default",axis=1)
y=df["Default"]

####  6. Model Building & Model Evaluation

##### 1) Logistic Regression

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression
model=LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)

In [ ]:
y_pred=model.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report,confusion_matrix
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

Although accuracy is high (~89%), the model performs poorly on the minority class with very low recall (~4%), making it unsuitable for credit risk prediction.

In [ ]:
# Handling Imbalance Data
model=LogisticRegression(max_iter=1000,class_weight="balanced")
model.fit(X_train,y_train)

In [ ]:
y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

After handling class imbalance, recall improved significantly from 4% to 68%, making the model more suitable for identifying high-risk customers, even though precision decreased.

##### 2) Random Forest

Because Captures complex patterns

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Train Decision Tree
dt = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced",
    max_depth=5   # control overfitting (important)
)

dt.fit(X_train, y_train)

In [ ]:
# Default prediction (threshold = 0.5)
y_pred_dt = dt.predict(X_test)
print("=== Default Threshold (0.5) ===")
print(confusion_matrix(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

In [ ]:
# Adjusting Threshold
y_probs_dt = dt.predict_proba(X_test)[:, 1]
y_pred_new = (y_probs_dt > 0.3).astype(int) # Be more strict in detecting risk”

print("\n=== Adjusted Threshold (0.3) ===")
print(confusion_matrix(y_test, y_pred_new))
print(classification_report(y_test, y_pred_new))

##### 3) XGBoost

In [ ]:
import xgboost
print(xgboost.__version__)

In [ ]:
from xgboost import XGBClassifier
xgb=XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=(len(y_train[y_train==0])/len(y_train[y_train==1])),
    random_state=42
)
xgb.fit(X_train,y_train)

In [ ]:
# Saving the model
import joblib
joblib.dump(xgb,"model.pkl")

In [ ]:
# Adjusting Threshold
y_probs_xgb=xgb.predict_proba(X_test)[:,1]
y_pred_xgb=(y_probs_xgb>0.3).astype(int)
print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

In [ ]:
# Balanced recall + precision
for t in [0.3, 0.4, 0.5, 0.6]: 
    y_pred_temp = (y_probs_xgb > t).astype(int)
    
    print(f"\nThreshold: {t}")
    print(confusion_matrix(y_test, y_pred_temp))
    print(classification_report(y_test, y_pred_temp))

In [ ]:
# Adjusting Threshold
y_probs_xgb=xgb.predict_proba(X_test)[:,1]
y_pred_xgb=(y_probs_xgb>0.4).astype(int)
print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))

At threshold 0.4, the model effectively identifies high-risk customers while reducing unnecessary rejection of low-risk applicants compared to more aggressive thresholds.

#### 7. Model Comparison & Selection
XGboost with a threshold of 0.4 as it provides a good balance between recall and precision, ensuring most defaulters are identified while limiting false positives.

#### 8. Model Interpretation (Feature Importance)

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb.feature_importances_
})
feature_importance = feature_importance.sort_values(by='Importance', ascending=False)
feature_importance.head(10)

The model heavily relies on financial stability indicators like income, employment, and dependents, while also capturing lender-side risk signals like interest rate

In [ ]:
feature_importance.head(10).plot(kind='barh', x='Feature', y='Importance')
plt.gca().invert_yaxis()
plt.show()

####  9. Business Insights
The model identified age, interest rate, income, and employment status as key drivers of default. This aligns with real-world credit risk logic, where financial stability and repayment burden play a crucial role.

#### 10. Conclusion:
This project developed a credit risk prediction model using machine learning to identify potential loan defaulters. After handling class imbalance and optimizing the classification threshold, the XGBoost model achieved strong recall, enabling effective identification of high-risk customers. Feature importance analysis revealed that factors such as age, income, interest rate, and employment status significantly influence default risk. This solution can help financial institutions minimize losses and improve lending decisions.